In [1]:
# =========================
# 0. 安装 / 导入
# =========================
!pip -q install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# =========================
# 1. 创建 SparkSession
# =========================
spark = SparkSession.builder \
    .appName("DSOTM_Sentiment_Baseline") \
    .getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", "8")

# =========================
# 2. 读取数据
# =========================
file_path = "/content/dsotm_reviews.csv"   # 如果路径不同，改这里
df = spark.read.csv(file_path, header=True, inferSchema=True)

print("原始数据：")
df.printSchema()
print("原始样本数：", df.count())
df.show(5, truncate=100)

# =========================
# 3. 数据清洗
#    - 保留 Review / Rating
#    - 删除缺失
#    - 去重
#    - 去除空文本
# =========================
df = df.select("Review", "Rating")

df = df.dropna(subset=["Review", "Rating"])
df = df.dropDuplicates(["Review", "Rating"])

df = df.withColumn("Review", F.trim(F.col("Review")))
df = df.filter(F.col("Review") != "")

# 确保 Rating 为 double
df = df.withColumn("Rating", F.col("Rating").cast("double"))
df = df.dropna(subset=["Rating"])

print("清洗后样本数：", df.count())

# =========================
# 4. 标签构造
#    根据你的文档，情感标签由 Rating 决定
#    注意：3.0 的边界规则文档提取不完整，所以做成可配置
# =========================
THREE_AS_POSITIVE = True

if THREE_AS_POSITIVE:
    df = df.withColumn(
        "Sentiment",
        F.when(F.col("Rating") >= 3.0, F.lit("positive")).otherwise(F.lit("negative"))
    )
else:
    df = df.withColumn(
        "Sentiment",
        F.when(F.col("Rating") > 3.0, F.lit("positive")).otherwise(F.lit("negative"))
    )

df.groupBy("Sentiment").count().show()

# =========================
# 5. 固定数据切分
#    为了可复现，不直接 randomSplit
#    而是先基于内容构造稳定随机列，再切分
# =========================
df = df.withColumn(
    "split_key",
    (F.abs(F.hash(F.col("Review"), F.col("Rating"))) % 100).cast("int")
)

train_df = df.filter(F.col("split_key") < 80)
test_df = df.filter(F.col("split_key") >= 80)

print("训练集大小：", train_df.count())
print("测试集大小：", test_df.count())

# =========================
# 6. 标签编码
# =========================
label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label",
    handleInvalid="skip"
)

# =========================
# 7. 文本预处理
#    - 小写
#    - 去掉非字母字符
#    - 分词
#    - 去停用词
# =========================
train_df = train_df.withColumn(
    "clean_text",
    F.lower(F.regexp_replace(F.col("Review"), r"[^a-zA-Z\s]", " "))
)

test_df = test_df.withColumn(
    "clean_text",
    F.lower(F.regexp_replace(F.col("Review"), r"[^a-zA-Z\s]", " "))
)

tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="tokens",
    pattern="\\s+",
    gaps=True
)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

# =========================
# 8. TF-IDF 特征
# =========================
vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=10000,
    minDF=2.0
)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

# =========================
# 9. 逻辑回归 Baseline
# =========================
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=[
    label_indexer,
    tokenizer,
    remover,
    vectorizer,
    idf,
    lr
])

# =========================
# 10. 训练模型
# =========================
model = pipeline.fit(train_df)

# =========================
# 11. 预测
# =========================
predictions = model.transform(test_df)

predictions.select("Review", "Rating", "Sentiment", "label", "prediction", "probability").show(10, truncate=100)

# =========================
# 12. 评估
# =========================
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)

print("Test Accuracy:", round(accuracy, 4))
print("Test F1:", round(f1, 4))

# =========================
# 13. 查看标签映射
# =========================
label_model = model.stages[0]
print("标签顺序（StringIndexer）:", label_model.labels)

# =========================
# 14. 保存固定切分与预测结果
# =========================
train_df.select("Review", "Rating", "Sentiment").coalesce(1).write.mode("overwrite").option("header", True).csv("/content/train_fixed_split")
test_df.select("Review", "Rating", "Sentiment").coalesce(1).write.mode("overwrite").option("header", True).csv("/content/test_fixed_split")

predictions.select(
    "Review", "Rating", "Sentiment", "label", "prediction"
).coalesce(1).write.mode("overwrite").option("header", True).csv("/content/test_predictions")

print("已保存：")
print("/content/train_fixed_split")
print("/content/test_fixed_split")
print("/content/test_predictions")


原始数据：
root
 |-- Review: string (nullable = true)
 |-- Rating: double (nullable = true)

原始样本数： 1548
+----------------------------------------------------------------------------------------------------+------+
|                                                                                              Review|Rating|
+----------------------------------------------------------------------------------------------------+------+
|"""More has been said about Dark Side of the Moon than will ever be necessary both positive and n...|   4.5|
|What can I possibly say about an album that not only means so much to so many and has influenced ...|   5.0|
|"You know for a band that spent several albums trying to atone for unceremoniously ousting its fo...|   2.0|
|"          Has finally clicked with me in full after 30 years absolutely smashing the previous re...|   4.0|
|"          So why are people afraid to say this isnt a masterpiece but is basically a well-crafte...|   4.5|
+-------------------